In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow_datasets as tfds
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.applications import VGG16
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo 
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models, regularizers

In [2]:
ds = fetch_ucirepo(id=360)

In [3]:
df = ds.data.features.copy()
df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


In [4]:
target = df["CO(GT)"]
# drop the target colmn and dat and time columns
features = df.drop(columns=["CO(GT)","Date", "Time"])

In [5]:
features.replace(-200, np.nan, inplace=True)  # Nan here are -200 so replcae them with np.nan
#features.fillna(features.mean(), inplace=True)  # fill the nan with mean of each column
features.fillna(features.mean(), inplace=True)  # fill the nan with mean of each column

In [6]:
target.replace(-200, np.nan, inplace=True)  # Nan here are -200 so replcae them with np.nan
#features.fillna(features.mean(), inplace=True)  # fill the nan with mean of each column
target.fillna(target.mean(), inplace=True)  # fill the nan with mean of each column

In [7]:
features.head()

,PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [8]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

In [9]:
# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
# Sequential: Input > Hidden Layer 1 > Hidden Layer 2 > Output
baseline_model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train_scaled.shape[1],)), # input layer with shape matching the number of features
    layers.Dense(128, activation="relu"),  # first hidden layer with 128 neurons and relu activation function
    layers.Dense(64, activation="relu"), # second hidden layer with 64 neurons and relu activation function
    layers.Dense(32, activation="relu"), # third hidden layer with 32 neurons and relu activation function
    layers.Dense(1) # output layer with 1 neuron - no activation function cause its regression (continuous output)
])

baseline_model.compile(
    optimizer="adam",   # Controls how weights are updated
    loss="mse", # what the model tries to minimize
    metrics=["mae"] # MAE (Mean Absolute Error) what we monitor, not optimize
)

baseline_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 128)            │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,033 (47.00 KB)

 Trainable params: 12,033 (47.00 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history_baseline = baseline_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2487 - mae: 0.3305 - val_loss: 0.2786 - val_mae: 0.3586
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2329 - mae: 0.3246 - val_loss: 0.2770 - val_mae: 0.3630
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2277 - mae: 0.3193 - val_loss: 0.2647 - val_mae: 0.3466
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2150 - mae: 0.3133 - val_loss: 0.2629 - val_mae: 0.3484
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2392 - mae: 0.3322 - val_loss: 0.2642 - val_mae: 0.3452
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2258 - mae: 0.3244 - val_loss: 0.2597 - val_mae: 0.3410
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2245 - mae: 0.3223 - val_loss: 0.2641 - val_mae: 0.3482
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2252 - mae: 0.3187 - val_loss: 0.2708 - val_mae: 0.3527
Epoch 9/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - lo

In [24]:
reg_model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train_scaled.shape[1],)),  # input layer
    layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.0001)),
    layers.Dropout(0.2),  # Drop 30% of neurons
    layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.0001)),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(0.0001)),
    layers.Dropout(0.2),
    layers.Dense(1)
])

reg_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

reg_model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_19 (Dense)                │ (None, 128)            │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,033 (47.00 KB)

 Trainable params: 12,033 (47.00 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
history_reg = reg_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)


Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 2.0090 - mae: 1.0519 - val_loss: 0.4594 - val_mae: 0.4969
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.7175 - mae: 0.5978 - val_loss: 0.3472 - val_mae: 0.4037
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5570 - mae: 0.5306 - val_loss: 0.3938 - val_mae: 0.4385
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5170 - mae: 0.5050 - val_loss: 0.3767 - val_mae: 0.4297
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5534 - mae: 0.5172 - val_loss: 0.3369 - val_mae: 0.4030
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4746 - mae: 0.4891 - val_loss: 0.4285 - val_mae: 0.4619
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4719 - mae: 0.4826 - val_loss: 0.3425 - val_mae: 0.4053
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4335 - mae: 0.4637 - val_loss: 0.3350 - val_mae: 0.3897
Epoch 9/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - l

In [26]:
from tensorflow.keras import callbacks
# callbacks to save the best model and stop early if no improvement

checkpoint_cb = callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

earlystop_cb = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


In [27]:
# train regularized model with callbacks
history_reg_cb = reg_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[checkpoint_cb, earlystop_cb],
    verbose=1
)


Epoch 1/20
183/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3832 - mae: 0.4188
Epoch 1: val_loss improved from inf to 0.29455, saving model to best_model.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3833 - mae: 0.4189 - val_loss: 0.2945 - val_mae: 0.3724
Epoch 2/20
176/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3992 - mae: 0.4293
Epoch 2: val_loss improved from 0.29455 to 0.29347, saving model to best_model.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3973 - mae: 0.4285 - val_loss: 0.2935 - val_mae: 0.3679
Epoch 3/20
184/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3628 - mae: 0.4132
Epoch 3: val_loss improved from 0.29347 to 0.28972, saving model to best_model.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3627 - mae: 0.4132 - val_loss: 0.2897 - val_mae: 0.3623
Epoch 4/20
184/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3858 - mae: 0.4133
Epoch 4: val_loss did not improve from 0.28972
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3858 -

In [28]:
# evaluate both models
baseline_loss, baseline_mae = baseline_model.evaluate(X_test_scaled, y_test, verbose=0)
reg_loss, reg_mae = reg_model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Baseline Model - Test Loss: {baseline_loss:.4f}, Test MAE: {baseline_mae:.4f}")
print(f"Regularized Model - Test Loss: {reg_loss:.4f}, Test MAE: {reg_mae:.4f}")

Baseline Model - Test Loss: 0.2419, Test MAE: 0.3254
Regularized Model - Test Loss: 0.2609, Test MAE: 0.3456
